# UK COVID-19 Trends: Admissions, Hospitalizations, Vaccinations, and Ventilation Beds


### Introduction 

Coronavirus disease 2019 (COVID-19) is a virus caused by the severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2). It was first identified in Wuhan, China, in 2019. Typical symptoms associated with the virus include fever, cough, headache, breathing difficulties, and loss of smell and taste. The symptoms of the disease usually become noticeable after an incubation period that ranges between 2 to 14 days, with the median time being around 4-5 days after exposure (Who.int, 2023).

The transmission of the virus primarily occurs through respiratory droplets and aerosols produced when an infected person coughs, sneezes, talks, or breathes. The risk of contamination significantly increases with the closeness of proximity between infected and uninfected individuals, especially in poorly ventilated and crowded spaces.

Mechanical ventilation beds have been a contentious issue for the UK government during the pre-pandemic and post-pandemic periods. The underlying argument stems from a comparative study between leading European nations, such as Germany and France, regarding the availability of mechanical ventilation beds per 1,000 patients. According to data from the King's Fund , the UK has 2.43 beds per 1,000 population. France and Germany boast 5.73 and 7.82 beds per 1,000 population, respectively (The King’s Fund, 2023).

This disparity in the availability of mechanical ventilation beds has led to concerns about the UK's preparedness and ability to handle a surge in COVID-19 cases, especially during the pandemic's peak. It has highlighted the importance of investing in healthcare infrastructure and resources to ensure countries are better equipped to manage future health crises.

With the datasets provided to me by Dr. Harry Yu, an attempt will be made to illustrate trends and correlated trends between three datasets - these being patient new admission, patients in hospital and vaccinations daily. Understanding the relationship between these factors is essential to make informed decisions regarding healthcare resource allocation, vaccination strategies, and public health policies.

In [ ]:
# All the necessary libraries are imported here
import pandas as pd
import numpy as np
# mdates
import matplotlib.dates as mdates
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from prophet import Prophet
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.ensemble import IsolationForest
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from matplotlib.dates import DateFormatter
import pandas as pd


#### EDA (Exploratory data analysis)

In [ ]:
# read data
df_deaths = pd.read_csv('data/Deaths.csv')

# drop duplicates
df_deaths = df_deaths.drop_duplicates()

# If there are duplicates print them
if df_deaths.duplicated().any():
    print("There are duplicates:")
    print(df_deaths[df_deaths.duplicated()])
else:
    print("There are no duplicates:")


# Check for missing values
if df_deaths.isnull().values.any():
    print("There are missing values:")
    print(df_deaths[df_deaths.isnull().any(axis=1)])
    # Drop missing values
    df_deaths = df_deaths.dropna()
df_deaths.tail()


In [ ]:

# read data
df_addmissions = pd.read_csv('data/PatientNewAdmissions.csv')

# drop duplicates
df_addmissions = df_addmissions.drop_duplicates()

# If there are duplicates print them
if df_addmissions.duplicated().any():
    print("There are duplicates:")
    print(df_addmissions[df_addmissions.duplicated()])
else:
    print("There are no duplicates:")


# Check for missing values
if df_addmissions.isnull().values.any():
    print("There are missing values:")
    print(df_addmissions[df_addmissions.isnull().any(axis=1)])
    # Drop missing values
    df_addmissions = df_addmissions.dropna()
df_addmissions.head()

# Description of the data
df_addmissions.describe()


In [ ]:
# read data
df_vaccinations = pd.read_csv('data/VaccinationsDaily.csv')

# drop duplicates
df_vaccinations = df_vaccinations.drop_duplicates()

# If there are duplicates print them
if df_vaccinations.duplicated().any():
    print("There are duplicates:")
    print(df_vaccinations[df_vaccinations.duplicated()])
else:
    print("There are no duplicates:")


# Check for missing values
if df_vaccinations.isnull().values.any():
    print("There are missing values:")
    print(df_vaccinations[df_vaccinations.isnull().any(axis=1)])
    # Drop missing values
    df_vaccinations = df_vaccinations.dropna()
# Check the last 5 rows
df_vaccinations.tail()

# Split


In [ ]:

# Set style
sns.set_style('whitegrid')
sns.set_palette('tab10')

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Box plot of daily admissions
sns.boxplot(x=df_addmissions['newAdmissions'], ax=axes[0])
axes[0].set_title('Box plot of daily admissions')
axes[0].set_xlabel('Number of admissions')
# grid true
axes[0].grid(True)

# Box plot of daily new deaths per 28 days
sns.boxplot(x=df_deaths['newDeaths28DaysByDeathDate'], ax=axes[1])
axes[1].set_title('Box plot of daily new deaths per 28 days')
axes[1].set_xlabel('Number of deaths per 28 days')
# grid true
axes[1].grid(True)

# Box plot of daily vaccinations
sns.boxplot(x=df_vaccinations['newPeopleVaccinatedFirstDoseByPublishDate'], ax=axes[2])
axes[2].set_title('Box plot of daily vaccinations')
axes[2].set_xlabel('Number of vaccinations')
# grid true
axes[2].grid(True)

# Display the plots
plt.tight_layout()
plt.show()

Some preprocessing is necessary for us to visually display the data. This process involves removing duplicates to prevent distortion of the analysis and potential misinterpretations. Both Deaths.csv and VaccinationsDaily.csv contained NaN values, which were eliminated to enhance the normalization of the dataset.

Further adjustments, such as reversing the data sets for plotting, are required. Conditionals were employed to verify the integrity of the dataset while executing the code. Alerts regarding missing values and duplicates were displayed on the screen, along with comments specifying whether duplicates or missing values had been removed.

The aforementioned box plots offer a visual representation of the skewness in the datasets. Each plot reveals a right-skewed distribution, which may be attributed to various social factors. For instance, the initial COVID-19 outbreak may have skewed the Deaths.csv data, while the availability of new vaccines and the easing of lockdown measures may have impacted the Admissions.csv data. These factors will be further examined within this notebook.